# 03 — Model Training

Train CNN (ResNet-18) and Siamese network models for
learned PRNU-based device identification.

Covers Experiments A1-A3, B1, B3.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt

from src.algorithms.cnn_classifier import (
    PRNUResNet, PRNUResNetLite, train_one_epoch, evaluate
)
from src.algorithms.siamese_network import (
    SiameseNetwork, train_siamese_epoch, evaluate_siamese
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

%matplotlib inline

In [ ]:
# Create synthetic training data
NUM_CLASSES = 5
NUM_SAMPLES = 200
PATCH_SIZE = 64

rng = np.random.RandomState(42)

# Simulate residual patches with class-specific patterns
X = np.zeros((NUM_SAMPLES, 1, PATCH_SIZE, PATCH_SIZE), dtype=np.float32)
y = np.zeros(NUM_SAMPLES, dtype=np.int64)

for i in range(NUM_SAMPLES):
    cls = i % NUM_CLASSES
    # Each class has a unique base pattern + noise
    pattern = rng.normal(0, 0.01, (PATCH_SIZE, PATCH_SIZE)) * (cls + 1)
    noise = rng.normal(0, 0.005, (PATCH_SIZE, PATCH_SIZE))
    X[i, 0] = pattern + noise
    y[i] = cls

# Split into train/val
n_train = int(0.8 * NUM_SAMPLES)
X_train, X_val = torch.FloatTensor(X[:n_train]), torch.FloatTensor(X[n_train:])
y_train, y_val = torch.LongTensor(y[:n_train]), torch.LongTensor(y[n_train:])

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=32, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val, y_val), batch_size=32)

print(f'Train: {len(X_train)} samples, Val: {len(X_val)} samples')

In [ ]:
# Train ResNet-18 CNN
model = PRNUResNet(num_classes=NUM_CLASSES, in_channels=1, pretrained=False).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

train_losses, val_accs = [], []
for epoch in range(20):
    train_metrics = train_one_epoch(model, train_loader, optimizer, device)
    val_metrics = evaluate(model, val_loader, device)
    train_losses.append(train_metrics['loss'])
    val_accs.append(val_metrics['accuracy'])
    if (epoch + 1) % 5 == 0:
        print(f'Epoch {epoch+1:2d} | Train Loss: {train_metrics["loss"]:.4f} | '
              f'Val Acc: {val_metrics["accuracy"]*100:.1f}%')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(train_losses); ax1.set_title('Training Loss'); ax1.set_xlabel('Epoch')
ax2.plot([a*100 for a in val_accs]); ax2.set_title('Validation Accuracy (%)')
ax2.set_xlabel('Epoch')
plt.tight_layout()
plt.show()

In [ ]:
# Compare patch sizes (Experiment B3)
for patch_size in [32, 64]:
    X_ps = np.zeros((NUM_SAMPLES, 1, patch_size, patch_size), dtype=np.float32)
    for i in range(NUM_SAMPLES):
        cls = i % NUM_CLASSES
        X_ps[i, 0] = rng.normal(0, 0.01 * (cls+1), (patch_size, patch_size))
    
    n_t = int(0.8 * NUM_SAMPLES)
    loader_t = DataLoader(TensorDataset(torch.FloatTensor(X_ps[:n_t]),
                          torch.LongTensor(y[:n_t])), batch_size=32, shuffle=True)
    loader_v = DataLoader(TensorDataset(torch.FloatTensor(X_ps[n_t:]),
                          torch.LongTensor(y[n_t:])), batch_size=32)
    
    m = PRNUResNetLite(NUM_CLASSES).to(device)
    opt = torch.optim.Adam(m.parameters(), lr=1e-3)
    for _ in range(20):
        train_one_epoch(m, loader_t, opt, device)
    val = evaluate(m, loader_v, device)
    print(f'Patch {patch_size}x{patch_size}: {val["accuracy"]*100:.1f}% accuracy')